# Đợt 2, account B: SDN + ABILENE (run 5–9)

Account B chạy SDN + ABILENE; account A chạy GÉANT bằng `kaggle_a.ipynb`.

- **GPU:** mỗi run ST-WaveFormer (epoch 200, patience 30) là một tiến trình, 1 run / GPU; GPU rảnh nhận run kế tiếp.
- **CPU:** song song với DL: 4 mô hình học máy (champion và LightGBM-Residual trước), rồi cache, stacking, ablation của từng run ngay khi run đó đủ 3 nhánh.
- **Push:** mỗi phần hoàn tất được commit và push lên nhánh `run_b` (clone từ `run`, có kết quả run 0–4 và champion đã cố định).
- **Kết thúc:** account xong trước dừng ngay để tiết kiệm quota. Account xong sau lấy kết quả của account kia, lập báo cáo run 0–9 và push lên `run`.
- **Bị ngắt:** chạy lại notebook; tự tiếp tục từ `run_b`, bỏ qua phần đã xong.

**Trước khi chạy:** Settings, Accelerator: **GPU T4 x2**; Internet: **On**; Add-ons, Secrets: `GITHUB_TOKEN` (quyền ghi repo). Chạy bằng **Save Version, Save & Run All**.

## 1. Cấu hình

In [ ]:
import os

ACCOUNT = "B"
PLAN    = {"A": ["geant"], "B": ["sdn", "abilene"]}   # dataset của từng account (thứ tự = thứ tự chạy)
RUN_IDS, FINAL_RUN_IDS = "5-9", "0-9"

REPO_SLUG   = "Vannt20/network_prediction_v2"
BASE_BRANCH = "run"      # kết quả đợt 1 + champion; cũng là nhánh nhận kết quả cuối
SECRET_NAME = "GITHUB_TOKEN"
GIT_USER_NAME, GIT_USER_MAIL = "Van207", "Nguyenthevan207@gmail.com"
WORKDIR = "/kaggle/working/network_prediction_v2"
BRANCH  = f"run_{ACCOUNT.lower()}"

## 2. Môi trường

So phiên bản torch/CUDA với output cùng cell ở account kia: phải giống nhau.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader
import platform, numpy, sklearn, torch
print(f"python {platform.python_version()} | torch {torch.__version__} (CUDA {torch.version.cuda}) | "
      f"numpy {numpy.__version__} | sklearn {sklearn.__version__} | CPU {os.cpu_count()}")
assert torch.cuda.device_count() >= 2, "Cần Accelerator = GPU T4 x2"

## 3. GitHub và mã nguồn

Token truyền qua header HTTP ở từng lệnh `git`, không ghi ra đĩa hay output.

In [ ]:
import base64, importlib.util, subprocess, sys
from kaggle_secrets import UserSecretsClient

_TOKEN = UserSecretsClient().get_secret(SECRET_NAME)
assert _TOKEN, f"Không đọc được secret {SECRET_NAME}"
_AUTH = base64.b64encode(f"x-access-token:{_TOKEN}".encode()).decode()
_AUTH_ARGS = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {_AUTH}"]


def git(*args, cwd=None, auth=False, check=True, quiet=True):
    r = subprocess.run(["git"] + (_AUTH_ARGS if auth else []) + list(args), cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_TOKEN, "***").replace(_AUTH, "***").strip()
    if (not quiet or r.returncode) and out:
        print(f"$ git {' '.join(args)}\n{out}")
    if check and r.returncode:
        raise RuntimeError(f"git {args[0]} lỗi (exit {r.returncode})")
    return r


git("config", "--global", "user.name", GIT_USER_NAME)
git("config", "--global", "user.email", GIT_USER_MAIL)
git("config", "--global", "http.postBuffer", "524288000")
REPO_URL = f"https://github.com/{REPO_SLUG}.git"

RESUMED = bool(git("ls-remote", "--heads", REPO_URL, BRANCH, auth=True).stdout.strip())
if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    git("clone", "-q", "--depth", "1", "--branch", BRANCH if RESUMED else BASE_BRANCH, REPO_URL, WORKDIR, auth=True)
os.chdir(WORKDIR)
git("checkout", "-q", "-B", BRANCH, cwd=WORKDIR)
print("Tiếp tục từ" if RESUMED else "Bắt đầu từ", BRANCH if RESUMED else BASE_BRANCH, "|",
      git("log", "--oneline", "-1", cwd=WORKDIR).stdout.strip())

missing = [p for p in ["lightgbm", "catboost", "xgboost"] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

## 4. Huấn luyện

Mỗi phút xóa màn hình và in một bảng trạng thái: việc trên từng GPU (epoch, epoch tốt nhất, patience, s/epoch, % GPU, VRAM),
làn ML, làn Ensemble, số phần đã xong và 10 sự kiện gần nhất. Log đầy đủ: `/kaggle/working/logs_kaggle/`.

In [ ]:
from training.kaggle_phase2 import Phase2

job = Phase2(ACCOUNT, PLAN, RUN_IDS, git, WORKDIR, FINAL_RUN_IDS, final_branch=BASE_BRANCH)
job.train()

## 5. Kết thúc phần của account B

Kiểm tra đủ kết quả, push, đánh dấu hoàn tất. Nếu account A chưa xong, notebook dừng ở đây; account A sẽ làm phần cuối.

In [ ]:
job.finish()

## 6. Phần cuối (chỉ account xong sau)

Lấy kết quả của account A, dựng lại bảng kết quả, tổng hợp ablation, lập báo cáo và hình cho run 0–9, push lên `run`.
Nhánh `phase2_final_lock` đảm bảo chỉ một account làm bước này; muốn làm lại thì xóa nhánh đó trên GitHub.

In [ ]:
if job.do_final:
    job.final()
    job.summary()
else:
    print("Phần cuối do account kia thực hiện.")

## 7. Kéo kết quả về máy

```bash
git fetch origin run
git switch run
git merge --ff-only origin/run
python preflight_check.py --run_ids 0-9
```